# Nested Cross-Validation – Coledocolitiasis · **v3**

> Sobre el v2. Cambios de esta versión:
>
> 1. **Modelo final = LR Regularizada (L1)**. Empata estadísticamente con RF (todos los p>0.05 salvo SVM) y se elige por **interpretabilidad** (coeficientes / odds ratios) y buena **estabilidad de hiperparámetros** (3/5 folds). RF queda como referencia.
> 2. **Recalibración con `CalibratedClassifierCV`** (Platt/sigmoid, ajustada dentro del inner CV de cada fold). El `class_weight='balanced'` infraestimaba la probabilidad de CDL+ (intercepto ~+1,5); la calibración lo corrige (Brier 0.22→0.14, intercepto ~0).
> 3. Matriz de confusión, umbrales, curvas y **net benefit** ahora sobre las **probabilidades calibradas** (imprescindible para que el net benefit sea válido).
> 4. **Nueva celda de interpretación**: coeficientes y odds ratios del modelo L1.
>
> La celda pesada sigue siendo el Nested CV (sección 6). El resto es instantáneo.

## 0. Dependencias

In [4]:
!pip install catboost imbalanced-learn xgboost lightgbm -q

## 1. Importaciones

In [5]:
import warnings, ast
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t as student_t, loguniform, uniform, randint, ttest_rel, wilcoxon

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils import shuffle, resample
from sklearn.model_selection import (StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, cross_val_predict)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             classification_report, brier_score_loss)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 123

## 2. Carga de datos, variable respuesta y desbalanceo

**Decisión de métrica (clave).** La clase positiva es CDL+ y supone el **82 %** de la cohorte. Ese desbalanceo es un *artefacto de enriquecimiento* (solo entran pacientes con sospecha), no la prevalencia poblacional real. Por eso:

- **Métrica principal = ROC-AUC** (invariante a la prevalencia, transportable).
- **AP = secundaria**, reportada con la advertencia de que depende de esta prevalencia.
- **No** damos la vuelta al positivo: mantener CDL+ como positivo conserva la pregunta clínica; el cambio a ROC ya resuelve el problema del AP sin alterar la narrativa.

In [6]:
df = pd.read_excel("BASE FINAL_.xlsx")            # nombre real del fichero (con guion bajo)
df = shuffle(df, random_state=RANDOM_STATE).reset_index(drop=True)
df = df.drop(columns=["Nº Historia"])             # identificador, sin poder predictivo

TARGET = "Confirmación de la presencia de coledocolitiasis"
SUSP   = "Coledocolitiasis"                        # sospecha clínica previa (comparador, objetivo 3)

y = (df[TARGET] == "Si").astype(int).values
# Guardamos la sospecha clínica antes de modelar (se mantiene como predictor en X)
suspicion = (df[SUSP] == "Si").astype(float).values   # con NaN donde falte

X = df.drop(columns=[TARGET])
numeric_features     = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Forma: {X.shape}")
print(f"Numéricas   ({len(numeric_features)}): {numeric_features}")
print(f"Categóricas ({len(categorical_features)}): {categorical_features}")

Forma: (639, 23)
Numéricas   (15): ['Edad en años al ingreso', 'Presión arterial sistólica media', 'Presión arterial diastólica media', 'Frecuencia cardiaca media', 'Bilirrubina total', 'Bilirrubina directa', 'ALT', 'AST', 'Fosfatasa alcalina', 'GGT', 'Lipasa', 'PCR', 'Leucocitos', 'Neutrofilos', 'Diámetro de la via biliar principal']
Categóricas (8): ['Sexo', 'Dolor abdominal', 'Ictericia mucocutánea', 'Temperatura > 37.8 ºC', 'Prueba de imagen', 'Colelitiasis', 'Coledocolitiasis', 'Colecistitis']


In [7]:
neg, pos = np.bincount(y)
scale_pos_weight_val = neg / pos
baseline_ap = pos / (neg + pos)

print(f"CDL- (negativa): {neg}  ({neg/(neg+pos)*100:.1f}%)")
print(f"CDL+ (positiva): {pos}  ({pos/(neg+pos)*100:.1f}%)")
print(f"scale_pos_weight (neg/pos): {scale_pos_weight_val:.3f}")
print(f"Baseline AP (= prevalencia positiva): {baseline_ap:.4f}")
print(f"Baseline ROC-AUC (azar): 0.5000   <- referencia honesta, no depende de la prevalencia")

CDL- (negativa): 115  (18.0%)
CDL+ (positiva): 524  (82.0%)
scale_pos_weight (neg/pos): 0.219
Baseline AP (= prevalencia positiva): 0.8200
Baseline ROC-AUC (azar): 0.5000   <- referencia honesta, no depende de la prevalencia


## 3. Transformers personalizados *(sin cambios respecto a v1)*

In [8]:
class ToCategory(BaseEstimator, TransformerMixin):
    """Convierte columnas a dtype 'category' para LightGBM nativo."""
    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            X[c] = X[c].astype("category")
        return X
    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return getattr(self, "feature_names_in_", np.array([], dtype=object))


class SklearnCatBoost(ClassifierMixin, BaseEstimator):
    """Wrapper sklearn-compatible para CatBoost (crea el objeto en fit para no romper clone)."""
    _estimator_type = "classifier"
    def __init__(self, cat_features=None, iterations=500, depth=6, learning_rate=0.05,
                 l2_leaf_reg=3, border_count=64, bagging_temperature=0.0,
                 auto_class_weights="Balanced", random_state=123):
        self.cat_features=cat_features; self.iterations=iterations; self.depth=depth
        self.learning_rate=learning_rate; self.l2_leaf_reg=l2_leaf_reg
        self.border_count=border_count; self.bagging_temperature=bagging_temperature
        self.auto_class_weights=auto_class_weights; self.random_state=random_state
    def fit(self, X, y):
        self.model_ = CatBoostClassifier(
            cat_features=self.cat_features, iterations=self.iterations, depth=self.depth,
            learning_rate=self.learning_rate, l2_leaf_reg=self.l2_leaf_reg,
            border_count=self.border_count, bagging_temperature=self.bagging_temperature,
            auto_class_weights=self.auto_class_weights, random_seed=self.random_state,
            verbose=0, allow_writing_files=False)
        self.model_.fit(X, y); self.classes_ = np.unique(y); return self
    def predict(self, X):       return self.model_.predict(X)
    def predict_proba(self, X): return self.model_.predict_proba(X)

print("Transformers definidos: ToCategory, SklearnCatBoost")

Transformers definidos: ToCategory, SklearnCatBoost


## 4. Preprocesadores por modelo *(sin cambios respecto a v1)*

In [9]:
# 1. OHE drop='first' + indicador + escalado  (LR, SVM, MLP)
pre_ohe_drop = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc",  StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                      ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore",
                                            sparse_output=False))]), categorical_features)]).set_output(transform="pandas")

# 2. OHE sin drop + indicador + escalado  (LR Regularizada)
pre_ohe_nodrop = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc",  StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                      ("ohe", OneHotEncoder(drop=None, handle_unknown="ignore",
                                            sparse_output=False))]), categorical_features)]).set_output(transform="pandas")

# 3. OHE drop='first' + indicador SIN escalado  (CART)
pre_cart = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                      ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore",
                                            sparse_output=False))]), categorical_features)]).set_output(transform="pandas")

# 4. OrdinalEncoder + indicador SIN escalado  (RF, ExtraTrees)
pre_trees = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                      ("ord", OrdinalEncoder(handle_unknown="use_encoded_value",
                                             unknown_value=-1))]), categorical_features)]).set_output(transform="pandas")

# 5. Passthrough numéricas + Ordinal  (XGBoost; aprende la dirección de NaN)
pre_xgb = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ord", OrdinalEncoder(handle_unknown="use_encoded_value",
                                             unknown_value=-1))]), categorical_features)]).set_output(transform="pandas")

# 6. Passthrough numéricas + Ordinal + ToCategory  (LightGBM nativo)
pre_lgbm = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", Pipeline([("imp",    SimpleImputer(strategy="most_frequent")),
                      ("ord",    OrdinalEncoder(handle_unknown="use_encoded_value",
                                                unknown_value=-1)),
                      ("to_cat", ToCategory())]), categorical_features)]).set_output(transform="pandas")

# 7. Passthrough numéricas + categóricas como strings  (CatBoost nativo)
pre_cb = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", SimpleImputer(strategy="most_frequent"), categorical_features)]).set_output(transform="pandas")
cat_features_cb = [f"cat__{c}" for c in categorical_features]

# 8. OHE sin indicador + escalado  (KNN; evita distorsionar distancias)
pre_knn = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore",
                                            sparse_output=False))]), categorical_features)]).set_output(transform="pandas")

print("8 preprocesadores definidos")

8 preprocesadores definidos


## 5. Modelos e hiperparámetros

**Único cambio funcional:** `PRIMARY_METRIC = 'roc_auc'`. El inner CV optimiza y refit sobre ROC-AUC. Los grids son los de Arturo.

In [10]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
PRIMARY_METRIC = "roc_auc"      # <<< CAMBIO v2 (antes 'average_precision')
N_ITER_RS = 50                  # baja a 10 para una pasada rápida

models = {
    "Logistic Regression": {
        "pipeline": Pipeline([("pre", pre_ohe_drop),
                              ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE,
                                                           class_weight="balanced"))]),
        "searcher": "grid",
        "params": [
            {"model__solver":["saga"], "model__penalty":["l2"],
             "model__C":[0.001,0.01,0.1,1,10,100]},
            {"model__solver":["saga"], "model__penalty":["l1"],
             "model__C":[0.001,0.01,0.1,1,10,100]},
            {"model__solver":["saga"], "model__penalty":["elasticnet"],
             "model__C":[0.01,0.1,1,10], "model__l1_ratio":[0.2,0.5,0.8]},
        ]},
    "LR Regularizada": {
        "pipeline": Pipeline([("pre", pre_ohe_nodrop),
                              ("model", LogisticRegression(penalty="l1", solver="saga",
                                                           max_iter=3000, random_state=RANDOM_STATE,
                                                           class_weight="balanced"))]),
        "searcher": "grid",
        "params": {"model__C":[1e-4,5e-4,1e-3,5e-3,0.01,0.05,0.1,0.5,1,5,10,50,100]}},
    "CART": {
        "pipeline": Pipeline([("pre", pre_cart),
                              ("model", DecisionTreeClassifier(random_state=RANDOM_STATE,
                                                               class_weight="balanced"))]),
        "searcher": "random",
        "params": {"model__criterion":["gini","entropy"], "model__max_depth":[3,5,7,10,None],
                   "model__min_samples_split":randint(2,25), "model__min_samples_leaf":randint(1,12),
                   "model__max_features":[None,"sqrt","log2"], "model__ccp_alpha":loguniform(1e-5,5e-2)}},
    "Random Forest": {
        "pipeline": Pipeline([("pre", pre_trees),
                              ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1,
                                                               class_weight="balanced_subsample"))]),
        "searcher": "random",
        "params": {"model__n_estimators":randint(100,600), "model__max_depth":[5,10,15,None],
                   "model__min_samples_split":randint(2,15), "model__min_samples_leaf":randint(1,8),
                   "model__max_features":["sqrt","log2"], "model__max_samples":uniform(0.6,0.4)}},
    "ExtraTrees": {
        "pipeline": Pipeline([("pre", pre_trees),
                              ("model", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1,
                                                             class_weight="balanced_subsample"))]),
        "searcher": "random",
        "params": {"model__n_estimators":randint(100,600), "model__max_depth":[5,10,15,None],
                   "model__min_samples_split":randint(2,15), "model__min_samples_leaf":randint(1,8),
                   "model__max_features":["sqrt","log2"]}},
    "XGBoost": {
        "pipeline": Pipeline([("pre", pre_xgb),
                              ("model", XGBClassifier(eval_metric="aucpr", random_state=RANDOM_STATE,
                                                      n_jobs=-1, scale_pos_weight=scale_pos_weight_val))]),
        "searcher": "random",
        "params": {"model__n_estimators":randint(100,600), "model__max_depth":randint(3,10),
                   "model__learning_rate":loguniform(0.005,0.3), "model__subsample":uniform(0.6,0.4),
                   "model__colsample_bytree":uniform(0.6,0.4), "model__min_child_weight":randint(1,8),
                   "model__gamma":loguniform(1e-5,1.0), "model__reg_alpha":loguniform(1e-5,5.0),
                   "model__reg_lambda":loguniform(0.1,10.0)}},
    "LightGBM": {
        "pipeline": Pipeline([("pre", pre_lgbm),
                              ("model", LGBMClassifier(verbose=-1, random_state=RANDOM_STATE, n_jobs=-1,
                                                       class_weight="balanced", categorical_feature="auto"))]),
        "searcher": "random",
        "params": {"model__n_estimators":randint(100,600), "model__learning_rate":loguniform(0.005,0.3),
                   "model__num_leaves":randint(15,150), "model__max_depth":[-1,5,10,15],
                   "model__min_child_samples":randint(5,60), "model__subsample":uniform(0.6,0.4),
                   "model__colsample_bytree":uniform(0.6,0.4), "model__reg_alpha":loguniform(1e-5,5.0),
                   "model__reg_lambda":loguniform(0.1,10.0)}},
    "CatBoost": {
        "pipeline": Pipeline([("pre", pre_cb),
                              ("model", SklearnCatBoost(cat_features=cat_features_cb,
                                                        auto_class_weights="Balanced",
                                                        random_state=RANDOM_STATE))]),
        "searcher": "random",
        "params": {"model__iterations":randint(200,1200), "model__depth":randint(4,9),
                   "model__learning_rate":loguniform(0.005,0.2), "model__l2_leaf_reg":loguniform(0.5,10.0),
                   "model__border_count":[32,64,128], "model__bagging_temperature":uniform(0,1.5)}},
    "SVM": {
        "pipeline": Pipeline([("pre", pre_ohe_drop),
                              ("model", SVC(probability=True, random_state=RANDOM_STATE,
                                            class_weight="balanced"))]),
        "searcher": "grid",
        "params": [
            {"model__kernel":["rbf"],     "model__C":[0.01,0.1,1,10,100], "model__gamma":["scale","auto",0.01,0.1]},
            {"model__kernel":["poly"],    "model__C":[0.1,1,10], "model__degree":[2,3], "model__gamma":["scale"]},
            {"model__kernel":["sigmoid"], "model__C":[0.1,1,10], "model__gamma":["scale","auto"]}]},
    "KNN": {
        "pipeline": ImbPipeline([("pre", pre_knn), ("smote", SMOTE(random_state=RANDOM_STATE)),
                                 ("model", KNeighborsClassifier())]),
        "searcher": "grid",
        "params": {"model__n_neighbors":[3,5,9,15,21,31], "model__weights":["uniform","distance"],
                   "model__metric":["euclidean","manhattan"], "smote__k_neighbors":[3,5]}},
    "MLP": {
        "pipeline": ImbPipeline([("pre", pre_ohe_drop), ("smote", SMOTE(random_state=RANDOM_STATE)),
                                 ("model", MLPClassifier(max_iter=500, random_state=RANDOM_STATE,
                                                         early_stopping=True, validation_fraction=0.1,
                                                         n_iter_no_change=15))]),
        "searcher": "random",
        "params": {"model__hidden_layer_sizes":[(64,),(128,),(256,),(512,),(128,64),(256,128),(512,256),(256,128,64)],
                   "model__activation":["relu","tanh"], "model__alpha":loguniform(1e-5,1.0),
                   "model__learning_rate_init":loguniform(1e-4,5e-2), "model__batch_size":[32,64,128],
                   "smote__k_neighbors":[3,5,7]}},
}
print(f"{len(models)} modelos definidos | métrica principal = {PRIMARY_METRIC}")

11 modelos definidos | métrica principal = roc_auc


## 6. Nested Cross-Validation

Igual que en v1, pero en cada fold externo guardamos **ROC-AUC y AP**. El inner CV optimiza ROC-AUC.

In [ ]:
def make_searcher(info):
    common = dict(estimator=info["pipeline"], cv=inner_cv,
                  scoring=PRIMARY_METRIC, n_jobs=-1, refit=True)
    if info["searcher"] == "grid":
        return GridSearchCV(param_grid=info["params"], **common)
    return RandomizedSearchCV(param_distributions=info["params"],
                              n_iter=N_ITER_RS, random_state=RANDOM_STATE, **common)

roc_scores  = {m: [] for m in models}
ap_scores   = {m: [] for m in models}
best_params = {m: [] for m in models}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for fold_i, (tr, te) in enumerate(outer_cv.split(X, y), 1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y[tr], y[te]
        for name, info in models.items():
            print(f"  Fold {fold_i} · {name}")
            s = make_searcher(info)
            s.fit(X_tr, y_tr)
            p = s.predict_proba(X_te)[:, 1]
            roc_scores[name].append(roc_auc_score(y_te, p))         # principal
            ap_scores[name].append(average_precision_score(y_te, p))# secundaria
            best_params[name].append(s.best_params_)
        print(f"Outer fold {fold_i}/5 completado.\n")
print("Nested CV completado.")

  Fold 1 · Logistic Regression
  Fold 1 · LR Regularizada
  Fold 1 · CART
  Fold 1 · Random Forest


## 7. Resultados — ranking por ROC-AUC (AP como referencia)

In [ ]:
model_order = sorted(roc_scores, key=lambda m: np.mean(roc_scores[m]), reverse=True)

def ic95(s):
    s = np.asarray(s); n = len(s); mu = s.mean(); sd = s.std(ddof=1)
    se = sd/np.sqrt(n); tc = student_t.ppf(0.975, df=n-1)
    return mu, sd, mu-tc*se, mu+tc*se

print(f"{'':>3}  {'Modelo':22}  {'ROC-AUC':>8}  {'IC95 ROC':>20}  {'AP':>7}")
print(f"{'':>3}  {'-'*22}  {'-'*8}  {'-'*20}  {'-'*7}")
for r, m in enumerate(model_order, 1):
    mu, sd, lo, hi = ic95(roc_scores[m])
    print(f"  {r:2d}. {m:22}  {mu:8.4f}  [{lo:.3f}, {hi:.3f}]  {np.mean(ap_scores[m]):7.4f}")
print(f"\n  Baseline ROC-AUC (azar) = 0.500 | Baseline AP (prevalencia) = {baseline_ap:.3f}")
print("  Nota: el AP parte de 0.82 por el desbalanceo; compárese siempre el ROC-AUC contra 0.5.")

In [ ]:
# Boxplot de ROC-AUC por modelo (ordenado)
fig, ax = plt.subplots(figsize=(10, 5))
data = [roc_scores[m] for m in model_order]
ax.boxplot(data, labels=model_order, showmeans=True)
ax.axhline(0.5, color="red", ls="--", lw=1, label="Azar (0.5)")
ax.set_ylabel("ROC-AUC (fold externo)"); ax.set_title("Comparación de modelos — Nested CV (ROC-AUC)")
plt.xticks(rotation=45, ha="right"); ax.legend(); plt.tight_layout(); plt.show()

### 7b. ¿El ganador es robusto? Comparación pareada del top-2

Con n=5 folds los IC se solapan. Antes de proclamar ganador, test pareado sobre los mismos folds.

In [ ]:
m1, m2 = model_order[0], model_order[1]
d = np.array(roc_scores[m1]) - np.array(roc_scores[m2])
t_stat, t_p = ttest_rel(roc_scores[m1], roc_scores[m2])
try:
    w_stat, w_p = wilcoxon(roc_scores[m1], roc_scores[m2])
except ValueError:
    w_p = np.nan
print(f"Top-1: {m1}  vs  Top-2: {m2}")
print(f"  Diferencia media ROC-AUC: {d.mean():+.4f}")
print(f"  Paired t-test  p = {t_p:.3f}")
print(f"  Wilcoxon       p = {w_p:.3f}" if not np.isnan(w_p) else "  Wilcoxon: n insuficiente")
print("\n  Si p > 0.05: no hay evidencia de que el top-1 sea mejor; elegir por parsimonia/interpretabilidad es legítimo.")

## 8. Modelo final y **predicciones OOF** (honestas)

El ganador se elige por programa. Las métricas y curvas se calculan sobre `cross_val_predict(outer_cv)`: cada paciente lo predice un modelo que **no** lo vio. Esto sustituye al `predict_proba(X)` in-sample de la v1.

In [ ]:
winner = "LR Regularizada"   # empate estadístico con RF; elegido por interpretabilidad + estabilidad
print(f"Modelo final: {winner}  (RF se mantiene como referencia)")

def norm(p):
    out = {}
    for k, v in p.items():
        if isinstance(v, np.floating): out[k] = float(v)
        elif isinstance(v, np.integer): out[k] = int(v)
        else: out[k] = v
    return out
modes = [tuple(sorted(norm(p).items())) for p in best_params[winner]]
mode_tuple, freq = Counter(modes).most_common(1)[0]
mode_params = dict(mode_tuple)
print(f"Moda de hiperparámetros ({freq}/5 folds): {mode_params}")

base_pipe = clone(models[winner]["pipeline"]).set_params(**mode_params)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    proba_raw = cross_val_predict(base_pipe, X, y, cv=outer_cv,
                                  method="predict_proba", n_jobs=-1)[:, 1]
    calibrated = CalibratedClassifierCV(base_pipe, method="sigmoid", cv=inner_cv)
    proba_oof  = cross_val_predict(calibrated, X, y, cv=outer_cv,
                                   method="predict_proba", n_jobs=-1)[:, 1]

print(f"\nROC-AUC OOF (calibrado): {roc_auc_score(y, proba_oof):.4f}  | sin calibrar: {roc_auc_score(y, proba_raw):.4f}")
print(f"AP OOF: {average_precision_score(y, proba_oof):.4f}  (baseline {baseline_ap:.3f})")
print("La calibración Platt es monótona -> ROC/AP apenas cambian; lo que cambia es la fiabilidad de la probabilidad.")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model = clone(models[winner]["pipeline"]).set_params(**mode_params).fit(X, y)

## 9. Umbrales, matriz de confusión y métricas — sobre OOF

Reportamos **sensibilidad y especificidad** (no solo precisión/recall) porque el objetivo clínico —evitar pruebas invasivas innecesarias— vive en la especificidad. Mostramos dos umbrales: el de Youden (equilibrio) y uno de **alta sensibilidad** (rule-out, recall ≥ 0.95).

In [ ]:
def metrics_at(thr, proba=proba_oof, yt=y):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    sens = tp/(tp+fn); spec = tn/(tn+fp)
    ppv = tp/(tp+fp) if (tp+fp) else 0; npv = tn/(tn+fn) if (tn+fn) else 0
    f1 = 2*ppv*sens/(ppv+sens) if (ppv+sens) else 0
    return dict(thr=thr, sens=sens, spec=spec, ppv=ppv, npv=npv, f1=f1,
                tn=tn, fp=fp, fn=fn, tp=tp)

# Umbral de Youden sobre OOF
fpr, tpr, thr_roc = roc_curve(y, proba_oof)
youden = thr_roc[np.argmax(tpr - fpr)]
# Umbral de alta sensibilidad (rule-out): el mayor umbral con recall >= 0.95
prec, rec, thr_pr = precision_recall_curve(y, proba_oof)
mask = rec[:-1] >= 0.95
thr_hs = thr_pr[mask].max() if mask.any() else 0.0

for label, thr in [("Youden (equilibrio)", youden),
                   ("Alta sensibilidad (recall>=0.95)", thr_hs),
                   ("Por defecto (0.5)", 0.5)]:
    mt = metrics_at(thr)
    print(f"{label:34} thr={mt['thr']:.3f} | sens={mt['sens']:.3f} spec={mt['spec']:.3f} "
          f"PPV={mt['ppv']:.3f} NPV={mt['npv']:.3f} F1={mt['f1']:.3f}")

# Matriz de confusión al umbral de Youden
mt = metrics_at(youden)
cm = np.array([[mt['tn'], mt['fp']], [mt['fn'], mt['tp']]])
fig, ax = plt.subplots(figsize=(4.6, 3.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["CDL- (pred)","CDL+ (pred)"],
            yticklabels=["CDL- (real)","CDL+ (real)"], ax=ax, cbar=False)
ax.set_title(f"Matriz de confusión OOF — {winner} (umbral Youden={youden:.3f})")
plt.tight_layout(); plt.show()

## 10. Curvas ROC y PR (OOF) + sospecha clínica como comparador

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
auc_oof = roc_auc_score(y, proba_oof)
axes[0].plot(fpr, tpr, color="steelblue", lw=2, label=f"{winner} (AUC={auc_oof:.3f})")
axes[0].plot([0,1],[0,1], "r--", label="Azar")
# punto de la sospecha clínica
msk = ~np.isnan(suspicion); ys = y[msk]; ss = suspicion[msk]
tn,fp,fn,tp = confusion_matrix(ys, ss.astype(int)).ravel()
sens_s, spec_s = tp/(tp+fn), tn/(tn+fp)
axes[0].scatter(1-spec_s, sens_s, color="darkorange", zorder=5, s=70,
                label=f"Sospecha clínica (sens={sens_s:.2f}, esp={spec_s:.2f})")
axes[0].set_xlabel("1 - Especificidad (FPR)"); axes[0].set_ylabel("Sensibilidad (TPR)")
axes[0].set_title("Curva ROC (OOF)"); axes[0].legend(loc="lower right")

# PR
axes[1].plot(rec, prec, color="steelblue", lw=2, label=f"{winner} (AP={average_precision_score(y,proba_oof):.3f})")
axes[1].axhline(baseline_ap, color="red", ls="--", label=f"Baseline (AP={baseline_ap:.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precisión")
axes[1].set_title("Curva PR (OOF) — referencia, no principal"); axes[1].legend(loc="lower left")
plt.tight_layout(); plt.show()

## 11. Calibración *(nuevo)*

Un modelo que prioriza pacientes por probabilidad necesita probabilidades fiables. **Aviso:** `class_weight`/SMOTE descalibran, así que si la pendiente se aleja de 1 conviene recalibrar (p. ej. `CalibratedClassifierCV`) como paso futuro.

In [ ]:
def cal_params(p):
    eps = 1e-6
    logit = np.log(np.clip(p, eps, 1-eps) / (1 - np.clip(p, eps, 1-eps)))
    rc = LogisticRegression().fit(logit.reshape(-1, 1), y)
    return rc.coef_[0, 0], rc.intercept_[0], brier_score_loss(y, p)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (tag, p) in zip(axes, [("Sin calibrar", proba_raw), ("Calibrado (Platt)", proba_oof)]):
    fpos, mpred = calibration_curve(y, p, n_bins=10, strategy="quantile")
    sl, ic, br = cal_params(p)
    ax.plot([0,1],[0,1], "r--", label="Calibración perfecta")
    ax.plot(mpred, fpos, "o-", color="steelblue", label=tag)
    ax.set_xlabel("Probabilidad predicha media"); ax.set_title(tag)
    ax.text(0.04, 0.74, f"Brier = {br:.3f}\nPendiente = {sl:.2f}\nIntercepto = {ic:+.2f}",
            transform=ax.transAxes, bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    ax.legend(loc="lower right", fontsize=8)
axes[0].set_ylabel("Fracción observada de CDL+")
plt.suptitle("Calibración (OOF) — antes vs después de recalibrar"); plt.tight_layout(); plt.show()

## 12. Curva de decisión / net benefit *(nuevo — cumple el objetivo 3)*

Net benefit a lo largo de las probabilidades-umbral, comparando el modelo con **tratar a todos**, **tratar a nadie** y la **sospecha clínica**. Es la forma honesta de saber si el modelo aporta utilidad clínica real y de compararlo con la estrategia tradicional.

In [ ]:
def net_benefit(yt, proba, pt):
    n = len(yt); pred = (proba >= pt).astype(int)
    tp = np.sum((pred==1)&(yt==1)); fp = np.sum((pred==1)&(yt==0))
    return tp/n - (fp/n)*(pt/(1-pt))

pts = np.linspace(0.01, 0.95, 95)
nb_model = [net_benefit(y, proba_oof, pt) for pt in pts]
prev = y.mean()
nb_all  = [prev - (1-prev)*(pt/(1-pt)) for pt in pts]   # tratar a todos
nb_none = [0]*len(pts)                                   # tratar a nadie
# sospecha clínica (decisión fija, no depende de pt): su net benefit a cada pt
nb_susp = []
for pt in pts:
    predp = ss  # 0/1 de la sospecha (subconjunto con sospecha no nula)
    tp = np.sum((predp==1)&(ys==1)); fp = np.sum((predp==1)&(ys==0)); n = len(ys)
    nb_susp.append(tp/n - (fp/n)*(pt/(1-pt)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pts, nb_model, color="steelblue", lw=2, label=f"Modelo ({winner})")
ax.plot(pts, nb_all,  color="gray", ls="--", label="Tratar a todos")
ax.plot(pts, nb_none, color="black", ls=":", label="Tratar a nadie")
ax.plot(pts, nb_susp, color="darkorange", lw=1.5, label="Sospecha clínica")
ax.set_xlabel("Probabilidad-umbral"); ax.set_ylabel("Net benefit")
ax.set_title("Curva de decisión (OOF)")
ax.set_ylim(min(0, np.min(nb_model)-0.02), prev+0.03); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Lectura: el modelo solo es útil donde su curva queda por ENCIMA de 'tratar a todos' y 'a nadie'.")

## 13. Importancia de variables — odds ratios con estabilidad bootstrap

Al ser el modelo final una regresión, la importancia natural son sus **coeficientes / odds ratios** (dan dirección y magnitud, y separan el efecto del *valor* del efecto de la *ausencia*). Pero un único ajuste L1 es inestable con colinealidad, así que remuestreamos (bootstrap estratificado) para obtener, por variable, la **frecuencia de selección** (cuántas veces L1 la mantiene) y el **IC95 del OR**. Solo se muestran las seleccionadas en ≥50 % de los remuestreos.

> **Cautelas**: (1) los OR de las numéricas son **por 1 desviación típica** (van estandarizadas); (2) en rojo, los **indicadores de ausencia**: si dominan, el modelo aprende en parte *qué pruebas se pidieron* (MNAR), no solo fisiología.

In [ ]:
N_BOOT = 500          # 300-1000; sube para IC mas finos
rng = np.random.RandomState(RANDOM_STATE)
feat_names = clone(base_pipe).fit(X, y).named_steps["pre"].get_feature_names_out()
boot = pd.DataFrame(0.0, index=range(N_BOOT), columns=feat_names)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for b in range(N_BOOT):
        idx = resample(np.arange(len(y)), replace=True, stratify=y,
                       random_state=rng.randint(1_000_000_000))
        fb = clone(base_pipe).fit(X.iloc[idx], y[idx])
        boot.loc[b, fb.named_steps["pre"].get_feature_names_out()] = fb.named_steps["model"].coef_[0]

OR = np.exp(boot)
imp = pd.DataFrame({"sel_%": (boot.abs() > 1e-8).mean()*100, "OR": OR.median(),
                    "IC_lo": OR.quantile(0.025), "IC_hi": OR.quantile(0.975)})
imp = imp[imp["sel_%"] >= 50].sort_values("sel_%")
nice = [i.replace("num__","").replace("cat__","").replace("missingindicator_","[falta] ") for i in imp.index]

print(imp.assign(variable=nice).set_index("variable").round(3).sort_values("sel_%", ascending=False).to_string())

fig, ax = plt.subplots(figsize=(8, len(imp)*0.36 + 1))
for j, (_, row) in enumerate(imp.iterrows()):
    col = "#d65f5f" if "[falta]" in nice[j] else "steelblue"
    ax.plot([row["IC_lo"], row["IC_hi"]], [j, j], color=col, lw=2)
    ax.plot(row["OR"], j, "o", color=col)
    ax.text(row["IC_hi"], j, f"  {row['sel_%']:.0f}%", va="center", fontsize=7, color="gray")
ax.axvline(1, color="k", ls="--", lw=0.8)
ax.set_yticks(range(len(imp))); ax.set_yticklabels(nice, fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("Odds ratio (IC95 bootstrap)  ·  rojo = indicador de ausencia  ·  % = frecuencia de selección")
ax.set_title(f"Importancia / efecto — {winner} (bootstrap, N={N_BOOT})")
plt.tight_layout(); plt.show()

## 14. Resumen para la memoria (v3)

1. **Métrica principal ROC-AUC** (AP secundaria con aviso de prevalencia). Reescribir 2.5.1.
2. **Modelo final: LR Regularizada (L1)**, por interpretabilidad y estabilidad (empate estadístico con RF). Reportar los **odds ratios** (seccion 13).
3. Todas las métricas, matriz de confusión y curvas **OOF** y **calibradas**. Incluir **sensibilidad y especificidad**.
4. **Calibración** (antes/después) y **curva de decisión / net benefit** + comparación con la **sospecha clínica** -> cumple el objetivo 3.
5. **Reconocer dos limitaciones**: (a) discriminación modesta (ROC~0.70); el modelo **no supera a "derivar a todos"** en el rango de uso; (b) buena parte de la señal son **indicadores de ausencia** (MNAR), no valores clínicos.
6. Conclusión honesta: pipeline riguroso, modelo interpretable y bien calibrado que **mejora el juicio clínico pero no está listo para descartar pacientes**; hace falta validación externa con negativos representativos.